In [1]:
import boto3
from datetime import datetime
import json
bedrock_runtime = boto3.client(service_name= "bedrock-runtime" , region_name = "us-east-1")
MODEL_ID = "amazon.nova-micro-v1:0"

In [4]:
system_prompt = """
 You are an assistant that summarizes music reviews for a record company.
 Here are examples:

 Review: The latest album by The New Wave Band is a masterpiece! Every track is a hit.
 Summary: Reviewer praises the latest album as a masterpiece with hit tracks.

 Review: I was disappointed with the new single; it lacked the energy of their previous work.
 Summary: Reviewer expresses disappointment, noting a lack of energy compared to previous work.
 """
user_input_review = "This EP is a solid effort with a few standout songs, though some tracks feel repetitive."

In [9]:
no_cache_payload = {
    "system":[
        {"text": system_prompt}
    ],
    "messages":[
        {"role": "user",
        "content":[{"text":user_input_review + "/n Summary"}]
        }
    ]
}
no_cache_response = bedrock_runtime.converse(
    modelId = MODEL_ID,
    system = no_cache_payload["system"],
    messages = no_cache_payload["messages"],
    
)
no_cache_output = no_cache_response['output']['message']['content'][0]['text']
no_cache_input_tokens = no_cache_response['usage']['inputTokens']
no_cache_output_tokens = no_cache_response['usage']['outputTokens']
no_cache_tokens = no_cache_input_tokens + no_cache_output_tokens

print("[No Caching] Generated Summary:")
print(no_cache_output)
print(f"Total Tokens Used (No Cache): {no_cache_tokens}")


[No Caching] Generated Summary:
Summary: Reviewer describes the EP as a solid effort with standout tracks, though some songs feel repetitive.
Total Tokens Used (No Cache): 128


In [13]:
cachePoint = {"cachePoint":{"type":"default"}}
cache_payload = {
    "system":[
       {"text":system_prompt},
         cachePoint,
    ],
   
    "messages":[
        {"role": "user","content":[{"text":user_input_review + "/n Summary"}]}
    ]
}
cache_response = bedrock_runtime.converse(
    modelId = MODEL_ID,
    system = cache_payload["system"],
    messages = cache_payload["messages"]
)
cache_output = cache_response['output']['message']['content'][0]['text']
cache_input_tokens = cache_response['usage']['inputTokens']
cache_output_tokens = cache_response['usage']['outputTokens']
cache_tokens = cache_input_tokens + cache_output_tokens

print("[ Caching] Generated Summary:")
print(cache_output)
print(f"Total Tokens Used (No Cache): {cache_tokens}")


[ Caching] Generated Summary:
Summary: Reviewer views the EP as a solid effort with a few standout songs, although some tracks feel repetitive.
Total Tokens Used (No Cache): 46
